# Ingestion report

**Environment:** [uv](https://docs.astral.sh/uv/) project venv from the repo README (`uv venv --python 3.11 .venv`, `uv sync --extra desi --extra dev`). In Jupyter, use kernel **Python (data-lake)** (`uv run python -m ipykernel install --user --name=data-lake --display-name "Python (data-lake)"`).

Summarise what is already on disk for a **data lake deployment**: catalogs (Parquet/HATS), spectra and cutouts (Zarr tiles), optional DESI batch-ingest checkpoints, and cross-match tables.

**Configuration** is resolved like the `dl-*` CLIs: optional explicit path in the next cell, else `$DATA_LAKE_CONFIG`, else walk upward from the current working directory for `lake_config.toml`.

In [18]:
from __future__ import annotations

import json
from pathlib import Path

import polars as pl
from IPython.display import display
import zarr

from data_lake.config import LakeConfig, LakeConfigNotFound
from data_lake.lake_registry import iter_catalog_surveys, load_lake_registry, registry_path

# Readable large counts in every report table (e.g. 28,425,963)
pl.Config.set_thousands_separator(",")

# Set to a concrete path to override discovery (otherwise None)
LAKE_CONFIG_TOML: str | None = None

# If True, open every spectrum/cutout tile to sum source counts (accurate but I/O heavy for huge lakes)
COUNT_ZARR_ROWS: bool = True

try:
    cfg = LakeConfig.discover(LAKE_CONFIG_TOML)
except LakeConfigNotFound as e:
    raise RuntimeError(
        "No lake_config.toml found. Create one with dl-init, export DATA_LAKE_CONFIG, "
        "or set LAKE_CONFIG_TOML in this cell."
    ) from e

ROOT = cfg.lake.root
print("Lake:", cfg.lake.name)
print("Root:", ROOT)
print("Config:", cfg.source_path)
print("HEALPix order (default):", cfg.partitioning.hats_order)

Lake: caspian
Root: /shared/caspian/data
Config: /home/sotiria/projects/caspian/lake_config.toml
HEALPix order (default): 5


## Helpers

In [19]:
def _safe_json(path: Path) -> dict | None:
    if not path.is_file():
        return None
    try:
        return json.loads(path.read_text())
    except json.JSONDecodeError:
        return None


def _child_survey_dirs(layer_root: Path) -> list[Path]:
    """Immediate subdirectories that look like a survey (contain tiles or a *info.json)."""
    if not layer_root.is_dir():
        return []
    out: list[Path] = []
    for p in sorted(layer_root.iterdir()):
        if not p.is_dir():
            continue
        if p.name == "crossmatch":
            continue
        marker = (
            any(p.glob("catalog_info.json"))
            or any(p.glob("spectrum_info.json"))
            or any(p.glob("cutout_info.json"))
            or any(p.glob("Norder=*"))
        )
        if marker:
            out.append(p)
    return out


def _catalog_survey_dirs(catalogs_root: Path) -> list[Path]:
    """All catalog surveys under ``catalogs/`` (excludes ``crossmatch/``).

    Uses :func:`iter_catalog_surveys` (``catalog_info.json`` or ``Norder=*``) and
    also includes directories with Parquet tiles or ingest checkpoints only
    (batch in progress or missing finalize).
    """
    if not catalogs_root.is_dir():
        return []
    by_name: dict[str, Path] = {}
    for name, path in iter_catalog_surveys(catalogs_root):
        by_name[name] = path
    for p in sorted(catalogs_root.iterdir()):
        if not p.is_dir() or p.name == "crossmatch" or p.name in by_name:
            continue
        if (
            any(p.rglob("Npix=*.parquet"))
            or (p / ".ingest_checkpoint.json").is_file()
            or (p / ".ingest_checkpoint.json.jsonl").is_file()
        ):
            by_name[p.name] = p
    return [by_name[n] for n in sorted(by_name)]


def _count_parquet_tiles(survey_dir: Path) -> int:
    return sum(1 for f in survey_dir.rglob("*.parquet") if f.name.startswith("Npix="))


def _hats_order_from_tree(survey_dir: Path) -> int | None:
    """Infer HEALPix order from ``Norder=<n>/`` directories when ``catalog_info.json`` is missing."""
    for p in sorted(survey_dir.glob("Norder=*")):
        if p.is_dir():
            try:
                return int(p.name.split("=", 1)[1])
            except ValueError:
                continue
    return None


def _pixel_sky_stats(hats_order: int | None, n_tiles: int) -> dict[str, float | None]:
    """Sky fraction and area for HEALPix pixels with ≥1 on-disk tile (nested, equal-area)."""
    if hats_order is None or n_tiles <= 0:
        return {"pixel_sky_frac": None, "pixel_sky_deg2": None}
    import healpy as hp

    nside = hp.order2nside(int(hats_order))
    n_pix_full = hp.nside2npix(nside)
    pix_area_deg2 = hp.nside2pixarea(nside, degrees=True)
    return {
        "pixel_sky_frac": n_tiles / n_pix_full,
        "pixel_sky_deg2": n_tiles * pix_area_deg2,
    }


def _parquet_row_count_from_tiles(survey_dir: Path) -> int:
    """Sum row counts from Parquet footers (fast; used when ``catalog_info.json`` is stale)."""
    import pyarrow.parquet as pq

    from data_lake.ingest.fits_to_parquet import _is_valid_parquet_tile

    total = 0
    for path in sorted(survey_dir.rglob("Npix=*.parquet")):
        if _is_valid_parquet_tile(path):
            total += pq.read_metadata(str(path)).num_rows
    return total


def _catalog_survey_stats(survey_dir: Path, info: dict) -> dict:
    """Merge ``catalog_info.json`` with on-disk tile counts (batch jobs killed before finalize)."""
    n_tiles = _count_parquet_tiles(survey_dir)
    rows_on_disk = _parquet_row_count_from_tiles(survey_dir) if n_tiles else 0
    info_rows = info.get("total_rows")
    # Prefer live tile count when metadata is missing or still zero after tiles exist.
    if n_tiles and (info_rows is None or (info_rows == 0 and rows_on_disk > 0)):
        total_rows = rows_on_disk
        rows_source = "parquet_tiles"
    else:
        total_rows = info_rows if info_rows is not None else rows_on_disk
        rows_source = "catalog_info" if info_rows is not None else ("parquet_tiles" if n_tiles else "none")

    hats_order = info.get("hats_order")
    if hats_order is None:
        hats_order = _hats_order_from_tree(survey_dir)

    total_columns = info.get("total_columns")
    if total_columns is None and n_tiles:
        import pyarrow.parquet as pq

        sample = next(survey_dir.rglob("Npix=*.parquet"), None)
        if sample is not None:
            total_columns = len(pq.read_schema(str(sample)))

    density = (
        float(total_rows) / n_tiles
        if total_rows is not None and n_tiles > 0
        else None
    )
    pixel_sky = _pixel_sky_stats(hats_order, n_tiles)

    return {
        "hats_order": hats_order,
        "total_rows": total_rows,
        "total_columns": total_columns,
        "n_parquet_tiles": n_tiles,
        "density": density,
        "pixel_sky_frac": pixel_sky["pixel_sky_frac"],
        "pixel_sky_deg2": pixel_sky["pixel_sky_deg2"],
        "rows_source": rows_source,
    }


def _iter_zarr_tiles(survey_dir: Path) -> list[Path]:
    return sorted(survey_dir.rglob("*.zarr"))


def _zarr_n_sources(tile: Path) -> int:
    store = zarr.storage.LocalStore(str(tile))
    g = zarr.open_group(store=store, mode="r", zarr_format=3)
    return int(g["source_id"].shape[0])


def _crossmatch_surveys(cross_root: Path) -> list[Path]:
    if not cross_root.is_dir():
        return []
    return sorted(p for p in cross_root.iterdir() if p.is_dir())


def _display_report(df: pl.DataFrame) -> None:
    """Show a survey summary table with thousands separators on numeric columns."""
    display(df)

## Catalogs (Parquet / HATS)

Survey directory names match the ``--survey`` flag (e.g. ``UNWISE_W1``, not ``UNWISE``). The index below lists **every** catalog under ``catalogs/``; the table adds row/tile/sky stats.

Row counts come from ``catalog_info.json`` when present; otherwise from Parquet tile footers (same logic as ingest finalize).

- **density** — ``total_rows / n_parquet_tiles`` (mean sources per tile)
- **healpix_sky_frac** — % of the full sphere in HEALPix pixels with ≥1 tile (``n_parquet_tiles / 12·4^N``)
- **healpix_sky_deg2** — area of those pixels (each pixel counts at full equal-area size)

### Available catalogs (index)

Sorted list of every survey directory discovered under ``catalogs/`` (including in-progress batch ingests with checkpoints but no ``catalog_info.json`` yet). Optional: ``shared/registry/surveys.parquet`` from ``dl-refresh-lake-registry``.

In [ ]:
cat_root = cfg.catalogs_root
catalog_dirs = _catalog_survey_dirs(cat_root)

print(f"{len(catalog_dirs)} catalog(s) under {cat_root}\n")
index_rows = []
for survey_dir in catalog_dirs:
    info = _safe_json(survey_dir / "catalog_info.json") or {}
    n_tiles = _count_parquet_tiles(survey_dir)
    ck = survey_dir / ".ingest_checkpoint.json"
    ck_jsonl = survey_dir / ".ingest_checkpoint.json.jsonl"
    n_ck = 0
    if ck_jsonl.is_file():
        n_ck = sum(1 for _ in ck_jsonl.open())
    elif ck.is_file():
        data = _safe_json(ck) or {}
        completed = data.get("completed")
        n_ck = len(completed) if isinstance(completed, list) else 0
    if n_tiles and info:
        status = "ingested"
    elif n_tiles:
        status = "tiles (no catalog_info)"
    elif n_ck:
        status = "checkpoint only"
    else:
        status = "stub"
    index_rows.append(
        {
            "survey": survey_dir.name,
            "status": status,
            "n_parquet_tiles": n_tiles,
            "checkpoint_files": n_ck,
            "hats_order": info.get("hats_order") or _hats_order_from_tree(survey_dir),
            "path": str(survey_dir),
        }
    )

_display_report(pl.DataFrame(index_rows).sort("survey"))

reg_p = registry_path(ROOT)
if reg_p.is_file():
    reg_cat = (
        pl.from_arrow(load_lake_registry(ROOT))
        .filter(pl.col("modality") == "catalog")
        .sort("survey")
    )
    print(f"\nLake registry ({reg_p.relative_to(ROOT)}): {reg_cat.height} catalog row(s)")
    with pl.Config(tbl_rows=reg_cat.height):
        display(reg_cat.select("survey", "hats_order", "total_rows", "n_columns", "path"))
else:
    print(f"\nNo lake registry at {reg_p} — optional: dl-refresh-lake-registry")

In [24]:
rows = []
for survey in catalog_dirs:
    info = _safe_json(survey / "catalog_info.json") or {}
    stats = _catalog_survey_stats(survey, info)
    rows.append(
        {
            "survey": survey.name,
            "hats_order": stats["hats_order"],
            "total_rows": stats["total_rows"],
            "total_columns": stats["total_columns"],
            "n_parquet_tiles": stats["n_parquet_tiles"],
            "density": int(stats["density"]) if stats["density"] is not None else None,
            "healpix_sky_frac": 100*round(stats["pixel_sky_frac"],2) if stats["pixel_sky_frac"] is not None else None,
            "healpix_sky_deg2": round(stats["pixel_sky_deg2"]) if stats["pixel_sky_deg2"] is not None else None,
            "rows_source": stats["rows_source"],
            "path": str(survey),
        }
    )

if rows:
    df = pl.DataFrame(rows).sort("survey")
    with pl.Config(tbl_rows=df.height):
        _display_report(df)
    stale = df.filter(
        (pl.col("n_parquet_tiles") > 0)
        & (pl.col("rows_source") == "parquet_tiles")
    )
    if stale.height:
        print(
            "Note: rows counted from Parquet tiles (not catalog_info.json) for surveys above — "
            "typical when a batch job wrote tiles but was killed before finalize, or "
            "catalog_info.json was never created. Re-run ingest finalize or "
            "dl-validate-catalog-ingest after the batch completes."
        )
else:
    print(f"No catalog surveys under {cat_root}")

survey,hats_order,total_rows,total_columns,n_parquet_tiles,density,healpix_sky_frac,healpix_sky_deg2,rows_source,path
str,i64,i64,i64,i64,i64,f64,i64,str,str
"""ALLWISE""",4,"358,241,126",301,"1,504","238,192",49.0,"20,197","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""ASSEF18_C75""",1,"20,907,127",17,47,"444,832",98.0,"40,394","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""ASSEF18_C75_extended""",1,"26,331",17,47,560,98.0,"40,394","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""ASSEF18_R90""",1,"4,543,530",17,47,"96,670",98.0,"40,394","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""ASSEF18_R90_extended""",1,"20,645",17,47,439,98.0,"40,394","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""DESI_DR1""",5,"28,425,963",142,"4,941","5,753",40.0,"16,588","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""EUCLID_DR1""",4,"26,852,670",197,16,"1,678,291",1.0,215,"""catalog_info""","""/shared/caspian/data/catalogs/…"
"""GALEX_GR67""",2,"58,661,796",372,48,"1,222,120",25.0,"10,313","""catalog_info""","""/shared/caspian/data/catalogs/…"
"""UNWISE_W1""",4,"632,516,050",32,"1,008","627,496",33.0,"13,536","""parquet_tiles""","""/shared/caspian/data/catalogs/…"


Note: rows counted from Parquet tiles (not catalog_info.json) for surveys above — typical when a batch job wrote tiles but was killed before finalize, or catalog_info.json was never created. Re-run ingest finalize or dl-validate-catalog-ingest after the batch completes.


## Spectra (Zarr tiles)

In [23]:
spec_rows = []
spec_root = cfg.spectra_root
for survey in _child_survey_dirs(spec_root):
    info = _safe_json(survey / "spectrum_info.json") or {}
    tiles = _iter_zarr_tiles(survey)
    n_sources = None
    if COUNT_ZARR_ROWS and tiles:
        n_sources = sum(_zarr_n_sources(t) for t in tiles)
    ck = survey / ".ingest_checkpoint.json"
    n_ck = None
    if ck.is_file():
        data = _safe_json(ck) or {}
        n_ck = len(data.get("completed", []))
    fail = survey / ".ingest_failures.jsonl"
    n_fail = sum(1 for _ in fail.open()) if fail.is_file() else 0
    hats_order = info.get("hats_order") or _hats_order_from_tree(survey)
    pixel_sky = _pixel_sky_stats(hats_order, len(tiles))
    spec_rows.append(
        {
            "survey": survey.name,
            "hats_order": hats_order,
            "n_zarr_tiles": len(tiles),
            "pixel_sky_frac": pixel_sky["pixel_sky_frac"],
            "pixel_sky_deg2": pixel_sky["pixel_sky_deg2"],
            "n_spectra_in_zarr": n_sources,
            "wavelength_mode": info.get("wavelength_mode"),
            "has_resolution": info.get("has_resolution"),
            "checkpoint_completed_files": n_ck,
            "ingest_failure_lines": n_fail,
            "path": str(survey),
        }
    )

if spec_rows:
    _display_report(pl.DataFrame(spec_rows))
else:
    print(f"No spectrum surveys under {spec_root}")

if not COUNT_ZARR_ROWS:
    print("Note: COUNT_ZARR_ROWS=False — enable it to sum spectra from each tile's source_id length.")

survey,hats_order,n_zarr_tiles,pixel_sky_frac,pixel_sky_deg2,n_spectra_in_zarr,wavelength_mode,has_resolution,checkpoint_completed_files,ingest_failure_lines,path
str,i64,i64,f64,f64,i64,str,bool,i64,i64,str
"""DESI_DR1""",5,"3,959",0.322184,"13,291.054166","15,163,514","""shared""",false,205,0,"""/shared/caspian/data/spectra/D…"


## Cutouts (Zarr tiles)

In [8]:
cut_rows = []
cut_root = cfg.cutouts_root
for survey in _child_survey_dirs(cut_root):
    info = _safe_json(survey / "cutout_info.json") or {}
    tiles = _iter_zarr_tiles(survey)
    n_sources = None
    if COUNT_ZARR_ROWS and tiles:
        n_sources = sum(_zarr_n_sources(t) for t in tiles)
    hats_order = info.get("hats_order") or _hats_order_from_tree(survey)
    pixel_sky = _pixel_sky_stats(hats_order, len(tiles))
    cut_rows.append(
        {
            "survey": survey.name,
            "hats_order": hats_order,
            "n_bands": info.get("n_bands"),
            "pixel_sky_frac": pixel_sky["pixel_sky_frac"],
            "pixel_sky_deg2": pixel_sky["pixel_sky_deg2"],
            "height": info.get("height"),
            "width": info.get("width"),
            "n_zarr_tiles": len(tiles),
            "n_cutouts_in_zarr": n_sources,
            "path": str(survey),
        }
    )

if cut_rows:
    _display_report(pl.DataFrame(cut_rows))
else:
    print(f"No cutout surveys under {cut_root}")

No cutout surveys under /shared/caspian/data/cutouts


## Cross-matches (`catalogs/crossmatch/`)

In [ ]:
xm_root = cat_root / "crossmatch"
xm_rows = []
for xm in _crossmatch_surveys(xm_root):
    info = _safe_json(xm / "catalog_info.json") or {}
    hats_order = info.get("hats_order") or _hats_order_from_tree(xm)
    n_tiles = _count_parquet_tiles(xm)
    pixel_sky = _pixel_sky_stats(hats_order, n_tiles)
    xm_rows.append(
        {
            "name": xm.name,
            "survey_a": info.get("survey_a"),
            "survey_b": info.get("survey_b"),
            "match_radius_arcsec": info.get("match_radius_arcsec"),
            "hats_order": hats_order,
            "n_parquet_tiles": n_tiles,
            "pixel_sky_frac": pixel_sky["pixel_sky_frac"],
            "pixel_sky_deg2": pixel_sky["pixel_sky_deg2"],
            "path": str(xm),
        }
    )

if xm_rows:
    _display_report(pl.DataFrame(xm_rows))
else:
    print(f"No cross-match tables under {xm_root}")

## Catalog ↔ spectra linkage (optional)

If the catalog has `_spectrum_index`, count how many rows point into an ingested spectrum tile (`_spectrum_index >= 0`). Requires DuckDB and can scan the full catalog.

In [ ]:
RUN_LINKAGE_STATS = False

if RUN_LINKAGE_STATS:
    from data_lake.io.catalog import CatalogAccessor

    link_rows = []
    for survey in _catalog_survey_dirs(cat_root):
        try:
            acc = CatalogAccessor(ROOT, survey.name)
        except Exception as exc:
            link_rows.append({"survey": survey.name, "error": str(exc)})
            continue
        cols = acc.columns
        if "_spectrum_index" not in cols:
            link_rows.append({"survey": survey.name, "with_spectrum_index": False})
            continue
        total = int(acc.query("SELECT COUNT(*) AS n FROM catalog")["n"][0])
        with_spec = int(
            acc.query(
                "SELECT COUNT(*) AS n FROM catalog WHERE _spectrum_index >= 0"
            )["n"][0]
        )
        link_rows.append(
            {
                "survey": survey.name,
                "catalog_rows": int(total),
                "rows_with_spectrum": int(with_spec),
                "fraction": float(with_spec) / int(total) if int(total) else None,
            }
        )
    _display_report(pl.DataFrame(link_rows))
else:
    print("Set RUN_LINKAGE_STATS=True to compute _spectrum_index coverage per catalog.")